# 🎯 Embedding Model Training: Zero to Hero — A Guided Lab

You've *used* embeddings throughout this course. This lab teaches how embedding models like
Sentence-BERT and OpenAI's `text-embedding-3` are actually **trained** — the loss functions and
learning procedures that shape raw vectors into a meaningful semantic space.

**Beginner-first.** Every chapter explains the *concept* and *math* before code. Prerequisite:
the PyTorch lab (autograd, training loop) and the Embeddings & Search lab.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. What "training an embedding" even means
2. Similarity targets: what should be close, what should be far
3. Contrastive loss (the core idea)
4. Triplet loss: anchor, positive, negative
5. In-batch negatives (how real models train efficiently)
6. Building a trainable embedding model in PyTorch
7. The training loop for embeddings
8. Evaluating embedding quality
9. Hard negative mining
10. 🏆 Capstone: train an embedding model on a real similarity task


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
torch.manual_seed(0)
print("PyTorch:", torch.__version__)

---
## Chapter 1 — What "Training an Embedding" Even Means

📖 **Theory.** In the Embeddings & Search lab, you built embeddings using TF-IDF and a hand-made
"concept map." Those are **fixed, hand-designed** — no learning involved. Real embedding models
(Sentence-BERT, OpenAI's embedding models) instead **learn** a mapping `text -> vector` from
data, via the same gradient-descent training loop from the PyTorch lab, but with a special
**loss function** designed for one goal: **similar things end up close, dissimilar things end up
far apart** in vector space.

🖼️ **Diagram — before vs after training**
```
 BEFORE training (random weights):        AFTER training (learned):
    "cat" •         • "refund"               "cat" •  • "kitten"      (near: similar meaning)
       "dog" •    •  "money"                        
    "kitten" •  • "bank"                                    • "refund"  • "money"
    (random, no structure)                     (near: related meaning) (far from "cat" cluster)
```

🧠 **Mental model.** Training an embedding model is like slowly nudging a giant pile of points
in space so that meaning becomes *geometry* — points that mean similar things get pulled
together, points that mean different things get pushed apart, one gradient step at a time.


In [ ]:
# The embedding MODEL is just a neural network that outputs a fixed-size vector per input.
# We'll build a toy version: turn short word-sequences into 4-dimensional vectors.
class ToyEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, output_dim=4):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.project = nn.Linear(embed_dim, output_dim)
    def forward(self, token_ids):
        # average the token embeddings, then project to the final embedding space
        x = self.token_embed(token_ids)          # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)                          # (batch, embed_dim) -- simple pooling
        x = self.project(x)                         # (batch, output_dim)
        return F.normalize(x, dim=-1)               # unit-length vectors (standard practice)

encoder = ToyEncoder(vocab_size=50, embed_dim=16, output_dim=4)
sample_input = torch.randint(0, 50, (2, 5))   # 2 sentences, 5 tokens each
output = encoder(sample_input)
print("output shape:", output.shape, "(2 sentences -> 2 embeddings of dim 4)")
print("each embedding is unit length:", torch.allclose(output.norm(dim=1), torch.ones(2), atol=1e-5))

### ✏️ Your Turn 1.1
In a comment, explain why the encoder ends with `F.normalize` (unit-length vectors) rather than
leaving embeddings at arbitrary scale.

In [ ]:
# your explanation


✅ **Solution**
```python
# Unit-length vectors make cosine similarity equivalent to a simple dot product
# (since cos_sim = dot(a,b)/(|a||b|), and |a|=|b|=1 after normalization).
# It also prevents the model from "cheating" by just making vectors longer instead
# of actually learning better directions/angles.
```

---
## Chapter 2 — Similarity Targets: What Should Be Close, What Should Be Far

📖 **Theory.** To train with gradient descent, you need **labeled pairs**: examples of things
that *should* be similar (positive pairs) and things that *should* be dissimilar (negative
pairs). Real embedding models are trained on huge datasets of such pairs — e.g., a question and
its correct answer (positive), a question and an unrelated answer (negative); or two paraphrases
of the same sentence (positive) vs. two unrelated sentences (negative).

🖼️ **Diagram — building training pairs**
```
 positive pair:  ("how do I reset my password", "forgot password steps")     -> should be CLOSE
 negative pair:  ("how do I reset my password", "what is your refund policy") -> should be FAR
```


In [ ]:
# A toy dataset of positive and negative pairs (simplified word-index "sentences")
vocab = ["how","do","i","reset","my","password","forgot","steps","what","is","your","refund","policy","cat","dog","kitten"]
w2i = {w:i for i,w in enumerate(vocab)}

def encode_sentence(words, max_len=5):
    ids = [w2i[w] for w in words][:max_len]
    ids += [0] * (max_len - len(ids))   # pad with token 0
    return ids

positive_pairs = [
    (["how","do","i","reset","password"], ["forgot","password","steps"]),
    (["cat"], ["kitten"]),
]
negative_pairs = [
    (["how","do","i","reset","password"], ["what","is","your","refund","policy"]),
    (["cat"], ["refund","policy"]),
]
print("example positive pair:", positive_pairs[0])
print("example negative pair:", negative_pairs[0])
print("\nencoded (padded to length 5):", encode_sentence(positive_pairs[0][0]))

### ✏️ Your Turn 2.1
Add one more positive pair (`"dog"` and something semantically related from the vocab, like
`["dog"]`/`["kitten"]`... pick something sensible) and one more negative pair.

In [ ]:
# your new pairs
new_positive = None
new_negative = None


✅ **Solution**
```python
new_positive = (["dog"], ["cat"])          # both animals -- reasonably related
new_negative = (["dog"], ["refund","policy"])   # unrelated concepts
```

---
## Chapter 3 — Contrastive Loss (the Core Idea)

📖 **Theory.** **Contrastive loss** directly encodes the training goal: for a **positive pair**,
minimize distance (or maximize similarity); for a **negative pair**, push similarity below a
**margin**. A common form:

```
L = y * (1 - sim)  +  (1-y) * max(0, sim - margin)
```
where `y=1` for positive pairs, `y=0` for negative pairs, `sim` is cosine similarity, and
`margin` is how far apart negatives must be pushed (further pushing beyond the margin gives no
extra reward — no need to push negatives to *literally* opposite vectors).

🖼️ **Diagram — the contrastive loss intuition**
```
 positive pair:  loss is LOW when sim is HIGH (close together)   -> pulls together
 negative pair:  loss is LOW when sim is BELOW margin             -> pushes apart (up to a point)
                 loss is ZERO once sim < margin (no more pushing needed)
```


In [ ]:
def contrastive_loss(emb1, emb2, label, margin=0.3):
    sim = F.cosine_similarity(emb1, emb2)                 # (batch,) similarity per pair
    positive_loss = label * (1 - sim)
    negative_loss = (1 - label) * torch.clamp(sim - margin, min=0)
    return (positive_loss + negative_loss).mean()

# demonstrate the loss behavior directly (no model needed yet)
emb_a = F.normalize(torch.tensor([[1.0, 0.0]]), dim=1)
emb_close = F.normalize(torch.tensor([[0.9, 0.1]]), dim=1)   # similar direction
emb_far = F.normalize(torch.tensor([[-1.0, 0.1]]), dim=1)    # opposite direction

loss_positive_close  = contrastive_loss(emb_a, emb_close, label=torch.tensor([1.0]))
loss_positive_far    = contrastive_loss(emb_a, emb_far,   label=torch.tensor([1.0]))
loss_negative_close  = contrastive_loss(emb_a, emb_close, label=torch.tensor([0.0]))
loss_negative_far    = contrastive_loss(emb_a, emb_far,   label=torch.tensor([0.0]))

print(f"positive pair, actually close:  loss={loss_positive_close.item():.3f}  (low -- good, matches target)")
print(f"positive pair, actually far:    loss={loss_positive_far.item():.3f}  (high -- bad, should be close but isn't)")
print(f"negative pair, actually close:  loss={loss_negative_close.item():.3f}  (high -- bad, should be far but isn't)")
print(f"negative pair, actually far:    loss={loss_negative_far.item():.3f}  (low -- good, matches target)")

⚡ **Pro tip.** The `margin` hyperparameter matters: too small and the model barely pushes
negatives apart (weak signal); too large and it wastes effort over-separating pairs that are
already far enough apart.

### ✏️ Your Turn 3.1
Compute `contrastive_loss` for a **negative** pair where the embeddings are only *slightly*
separated (cosine similarity around 0.5, above the margin of 0.3). Confirm the loss is
**positive** (the model still needs to push them further apart).

In [ ]:
emb_b = F.normalize(torch.tensor([[0.5, 0.87]]), dim=1)   # roughly 60 degrees from emb_a
loss_slightly_separated = None
print(loss_slightly_separated)

✅ **Solution**
```python
loss_slightly_separated = contrastive_loss(emb_a, emb_b, label=torch.tensor([0.0]), margin=0.3)
print(loss_slightly_separated.item())   # > 0, since their similarity (~0.5) still exceeds margin (0.3)
```

---
## Chapter 4 — Triplet Loss: Anchor, Positive, Negative

📖 **Theory.** **Triplet loss** is contrastive loss's more common cousin in practice. Instead of
labeled pairs, you use **triplets**: an **anchor** (reference item), a **positive** (similar to
the anchor), and a **negative** (dissimilar). The loss directly compares the anchor-positive
distance to the anchor-negative distance:

```
L = max(0, d(anchor, positive) - d(anchor, negative) + margin)
```
This says: "the negative must be **at least `margin` further** from the anchor than the positive
is" — a *relative*, not absolute, requirement.

🖼️ **Diagram — the triplet relationship**
```
              margin
              ◄────►
 anchor ●──────────●  positive   (should be CLOSE)
    │
    └───────────────────────●  negative   (should be FURTHER than positive + margin)
```


In [ ]:
def triplet_loss(anchor, positive, negative, margin=0.3):
    # use squared Euclidean distance (common in triplet loss; cosine also works)
    d_pos = (anchor - positive).pow(2).sum(dim=1)
    d_neg = (anchor - negative).pow(2).sum(dim=1)
    return torch.clamp(d_pos - d_neg + margin, min=0).mean()

anchor = F.normalize(torch.tensor([[1.0, 0.0]]), dim=1)
positive = F.normalize(torch.tensor([[0.9, 0.1]]), dim=1)   # close to anchor -- good
negative = F.normalize(torch.tensor([[0.85, 0.15]]), dim=1) # also fairly close -- BAD, too similar to anchor

loss = triplet_loss(anchor, positive, negative)
print(f"triplet loss (negative too close to anchor): {loss.item():.4f}  (positive -- model needs to push negative away more)")

better_negative = F.normalize(torch.tensor([[-1.0, 0.1]]), dim=1)   # truly far from anchor
loss2 = triplet_loss(anchor, positive, better_negative)
print(f"triplet loss (negative properly far): {loss2.item():.4f}  (zero -- constraint already satisfied)")

⚠️ **Common trap.** If your negative is chosen **randomly** from a huge, diverse dataset, it's
often trivially far from the anchor already — giving **zero loss, zero learning signal**. This
motivates **hard negative mining** (Chapter 9): deliberately picking negatives that are
*confusingly close* to force real learning.

### ✏️ Your Turn 4.1
Compute triplet loss for an anchor/positive pair that's actually **far apart** (positive isn't
very positive) paired with a negative that's properly far. Confirm the loss is positive (the
model needs to improve the positive pair's closeness).

In [ ]:
weak_positive = F.normalize(torch.tensor([[0.0, 1.0]]), dim=1)   # 90 degrees from anchor -- not close at all
loss3 = None
print(loss3)

✅ **Solution**
```python
loss3 = triplet_loss(anchor, weak_positive, better_negative)
print(loss3.item())   # > 0 -- even with negative properly far, d(anchor,positive) is too large
```

---
## Chapter 5 — In-Batch Negatives (How Real Models Train Efficiently)

📖 **Theory.** Manually curating explicit negative pairs for every example doesn't scale.
**In-batch negatives** is the clever trick real models use: within a training **batch** of N
positive pairs, treat **every other pair's partner** as a negative for free. For a batch of N
positive pairs, you get N positives and N×(N-1) negatives — **without labeling a single extra
negative example**.

🖼️ **Diagram — in-batch negatives**
```
 batch of positive pairs:  (q1,a1), (q2,a2), (q3,a3)

 for q1: a1 is the positive; a2 and a3 automatically become negatives (they answer q2, q3, not q1)
 for q2: a2 is the positive; a1 and a3 automatically become negatives
 for q3: a3 is the positive; a1 and a2 automatically become negatives
```


In [ ]:
def in_batch_contrastive_loss(query_embs, doc_embs, temperature=0.1):
    # similarity matrix: every query vs every doc in the batch
    sims = query_embs @ doc_embs.T / temperature     # (batch, batch)
    # the CORRECT match for query i is doc i -- the diagonal!
    labels = torch.arange(len(query_embs))
    return F.cross_entropy(sims, labels)   # softmax + cross-entropy over each row

# 3 query-document positive pairs in a batch (toy 4-dim embeddings, already normalized)
torch.manual_seed(1)
queries = F.normalize(torch.randn(3, 4), dim=1)
docs = F.normalize(torch.randn(3, 4), dim=1)

loss = in_batch_contrastive_loss(queries, docs)
print("in-batch contrastive loss:", loss.item())
print("\nsimilarity matrix (diagonal = intended positive pairs):")
print((queries @ docs.T).detach().numpy().round(3))

⚡ **Pro tip.** This is exactly why embedding-model training uses **large batch sizes** —
bigger batches mean more free negatives per step, giving a stronger and more efficient training
signal. It's treating a batch's own structure as the labels (the diagonal is "correct").

### ✏️ Your Turn 5.1
Build a batch where `queries[0]` and `docs[0]` are made to be **very** similar (nearly identical
vectors) and see how the loss changes compared to random embeddings.

In [ ]:
queries2 = F.normalize(torch.randn(3, 4), dim=1)
docs2 = queries2.clone()   # docs2[0] identical direction to queries2[0], etc
docs2 = docs2 + 0.01*torch.randn(3,4)   # tiny perturbation
docs2 = F.normalize(docs2, dim=1)
loss2 = None
print(loss2)

✅ **Solution**
```python
loss2 = in_batch_contrastive_loss(queries2, docs2)
print(loss2.item())   # much LOWER than the random case -- diagonal similarities are now high
```

---
## Chapter 6 — Building a Trainable Embedding Model in PyTorch

📖 **Theory.** Let's assemble a complete trainable embedding model using the `ToyEncoder` from
Chapter 1 and the in-batch contrastive loss from Chapter 5 — this is structurally identical to
how real Sentence-BERT-style models are trained, just at toy scale.


In [ ]:
torch.manual_seed(2)

vocab = ["how","do","i","reset","my","password","forgot","steps",
         "what","is","your","refund","policy","cat","dog","kitten",
         "weather","today","sunny","rain"]
w2i = {w:i for i,w in enumerate(vocab)}

def encode(words, max_len=5):
    ids = [w2i.get(w,0) for w in words][:max_len]
    ids += [0]*(max_len-len(ids))
    return ids

# training data: (query, matching_document) positive pairs
training_pairs = [
    (["how","do","i","reset","password"], ["forgot","password","steps"]),
    (["what","is","your","refund","policy"], ["refund","policy","is"]),
    (["cat"], ["kitten"]),
    (["dog"], ["cat"]),
    (["weather","today"], ["sunny","rain"]),
]

model = ToyEncoder(vocab_size=len(vocab), embed_dim=16, output_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

query_ids = torch.tensor([encode(q) for q,_ in training_pairs])
doc_ids = torch.tensor([encode(d) for _,d in training_pairs])
print("query batch shape:", query_ids.shape, " doc batch shape:", doc_ids.shape)

### ✏️ Your Turn 6.1
Run `model(query_ids)` and `model(doc_ids)` (before any training) and print the resulting
similarity matrix. It should look mostly random/unstructured at this point.

In [ ]:
query_embs_before = None
doc_embs_before = None
print((query_embs_before @ doc_embs_before.T).detach().numpy().round(3) if query_embs_before is not None else None)

✅ **Solution**
```python
query_embs_before = model(query_ids)
doc_embs_before = model(doc_ids)
print((query_embs_before @ doc_embs_before.T).detach().numpy().round(3))
# diagonal isn't notably higher than off-diagonal yet -- untrained
```

---
## Chapter 7 — The Training Loop for Embeddings

📖 **Theory.** Same 5-step rhythm as the PyTorch lab: zero_grad → forward → loss → backward →
step. The only difference from a classifier is the **loss function** (in-batch contrastive
instead of cross-entropy on class labels) and that we run **two** forward passes per step (one
for queries, one for docs) since both feed into the same shared encoder.

🖼️ **Diagram — the embedding training loop**
```
 batch of (query, doc) pairs
        │
   encode both with the SAME model  (shared weights -- one encoder for everything)
        │
   in-batch contrastive loss (diagonal = correct matches)
        │
   backward + step  -->  embeddings for TRUE pairs get pulled closer,
                          embeddings for batch-mismatched pairs get pushed apart
```


In [ ]:
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    q_embs = model(query_ids)
    d_embs = model(doc_ids)
    loss = in_batch_contrastive_loss(q_embs, d_embs, temperature=0.2)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

print(f"\nfinal loss: {losses[-1]:.4f}  (started at {losses[0]:.4f})")

### ✏️ Your Turn 7.1
After training, compute the similarity matrix again (`model(query_ids) @ model(doc_ids).T`) and
confirm the **diagonal** values are now clearly higher than the off-diagonal ones — the model
has learned to match each query to its true document.

In [ ]:
with torch.no_grad():
    q_final = model(query_ids)
    d_final = model(doc_ids)
sim_matrix = None
print(sim_matrix)

✅ **Solution**
```python
with torch.no_grad():
    sim_matrix = (q_final @ d_final.T).numpy().round(3)
print(sim_matrix)
# diagonal entries should now be noticeably higher than off-diagonal ones
```

---
## Chapter 8 — Evaluating Embedding Quality

📖 **Theory.** After training, evaluate the same way you did in the Embeddings & Search lab:
**retrieval accuracy** — for each query, does its true matching document rank #1 (or in the
top-k) among all candidates by cosine similarity?

🖼️ **Diagram — retrieval-style evaluation**
```
 for each query: rank ALL documents by similarity
 correct if the TRUE matching doc is ranked #1 (top-1 accuracy)
 or within the top-k (top-k accuracy) -- same idea as recall@k from the Vector Search lab
```


In [ ]:
def top1_accuracy(query_embs, doc_embs):
    sims = query_embs @ doc_embs.T
    predicted = sims.argmax(dim=1)
    correct = torch.arange(len(query_embs))
    return (predicted == correct).float().mean().item()

with torch.no_grad():
    acc = top1_accuracy(model(query_ids), model(doc_ids))
print(f"top-1 retrieval accuracy after training: {acc:.2%}")

# compare to an UNTRAINED model for contrast
untrained = ToyEncoder(vocab_size=len(vocab), embed_dim=16, output_dim=8)
with torch.no_grad():
    acc_untrained = top1_accuracy(untrained(query_ids), untrained(doc_ids))
print(f"top-1 retrieval accuracy, untrained model: {acc_untrained:.2%}")

### ✏️ Your Turn 8.1
In a comment, explain why comparing to an "untrained model" baseline (rather than just reporting
the trained accuracy alone) is good practice.

In [ ]:
# your explanation


✅ **Solution**
```python
# A raw accuracy number is hard to interpret alone -- is 80% good or bad? Comparing
# to an untrained (random) baseline shows how much LEARNING actually contributed,
# isolating the model's real improvement from what randomness alone would achieve.
```

---
## Chapter 9 — Hard Negative Mining

📖 **Theory.** Random negatives (like in-batch negatives from unrelated pairs) are often "easy"
— already far apart, giving weak learning signal (as you saw the triplet-loss trap in Chapter 4).
**Hard negative mining** deliberately finds negatives that are **confusingly similar** to the
anchor but still technically wrong — these force the model to learn finer distinctions.

🖼️ **Diagram — easy vs hard negatives**
```
 anchor: "how do I reset my password"

 EASY negative:  "what is your refund policy"     (already very different -- weak signal)
 HARD negative:  "how do I reset my email"        (superficially similar wording,
                                                     but wrong answer -- forces real learning)
```


In [ ]:
def mine_hard_negatives(anchor_emb, candidate_embs, true_idx, top_k=1):
    """Find candidates MOST similar to the anchor, excluding the true match --
    these are the 'confusing' near-misses that make the best hard negatives."""
    sims = anchor_emb @ candidate_embs.T
    sims[0, true_idx] = -float("inf")   # exclude the true positive itself
    hard_idx = sims.argsort(descending=True)[0, :top_k]
    return hard_idx

with torch.no_grad():
    all_doc_embs = model(doc_ids)
    query0_emb = model(query_ids[0:1])   # "how do i reset password" query
    hard_neg_idx = mine_hard_negatives(query0_emb, all_doc_embs, true_idx=0, top_k=1)

print("query:", training_pairs[0][0])
print("true positive doc:", training_pairs[0][1])
print("hardest negative found (most confusing wrong match):", training_pairs[hard_neg_idx.item()][1])

⚡ **Pro tip.** Real production pipelines mine hard negatives **periodically during
training** (not just once) using the model's *current* embeddings — as the model improves, what
counts as "confusingly similar" shifts, so the hard-negative set should be refreshed.

### ✏️ Your Turn 9.1
Mine the **top-2** hardest negatives for `query_ids[2]` (the "cat"/"kitten" pair) instead of
just the top-1.

In [ ]:
with torch.no_grad():
    query2_emb = model(query_ids[2:3])
hard_negs_2 = None
print(hard_negs_2)

✅ **Solution**
```python
with torch.no_grad():
    hard_negs_2 = mine_hard_negatives(query2_emb, all_doc_embs, true_idx=2, top_k=2)
print([training_pairs[i][1] for i in hard_negs_2.tolist()])
```

---
## 🏆 Chapter 10 — Capstone: Train an Embedding Model on a Real Similarity Task

Build and train an embedding model on an **expanded** dataset of query-document pairs (support
tickets → categories, similar to the RAG lab's corpus), then evaluate its retrieval accuracy
before and after training. This mirrors exactly how a real domain-specific embedding model gets
fine-tuned.

In [ ]:
torch.manual_seed(3)

vocab2 = ["how","do","i","reset","my","password","forgot","login","credentials",
          "refund","money","back","cancel","subscription","order",
          "app","crash","freeze","slow","update",
          "payment","billing","address","method","change",
          "package","shipping","arrived","track","delivery"]
w2i2 = {w:i for i,w in enumerate(vocab2)}

def encode2(words, max_len=6):
    ids = [w2i2.get(w,0) for w in words][:max_len]
    ids += [0]*(max_len-len(ids))
    return ids

capstone_pairs = [
    (["how","do","i","reset","password"], ["forgot","login","credentials"]),
    (["refund","my","money"], ["cancel","subscription","order"]),
    (["app","keeps","crash"], ["app","freeze","slow"]),
    (["update","payment","method"], ["billing","address","change"]),
    (["where","is","my","package"], ["shipping","arrived","track"]),
]
print(f"{len(capstone_pairs)} training pairs across 5 support-ticket categories")

### ✏️ Capstone Tasks
1. Build a `ToyEncoder` for `vocab2` with `embed_dim=32, output_dim=16`.
2. Encode all queries and docs from `capstone_pairs` into tensors.
3. Measure **top-1 accuracy BEFORE** training (baseline).
4. Train for 300 epochs using in-batch contrastive loss.
5. Measure **top-1 accuracy AFTER** training and report the improvement.

In [ ]:
# Your training pipeline here


✅ **Capstone Solution**
```python
# 1. build model
cap_model = ToyEncoder(vocab_size=len(vocab2), embed_dim=32, output_dim=16)
cap_optimizer = torch.optim.Adam(cap_model.parameters(), lr=0.01)

# 2. encode data
cap_query_ids = torch.tensor([encode2(q) for q,_ in capstone_pairs])
cap_doc_ids = torch.tensor([encode2(d) for _,d in capstone_pairs])

# 3. accuracy BEFORE training
with torch.no_grad():
    acc_before = top1_accuracy(cap_model(cap_query_ids), cap_model(cap_doc_ids))

# 4. train
for epoch in range(300):
    cap_optimizer.zero_grad()
    q_e = cap_model(cap_query_ids)
    d_e = cap_model(cap_doc_ids)
    loss = in_batch_contrastive_loss(q_e, d_e, temperature=0.15)
    loss.backward()
    cap_optimizer.step()

# 5. accuracy AFTER training
with torch.no_grad():
    acc_after = top1_accuracy(cap_model(cap_query_ids), cap_model(cap_doc_ids))

print(f"top-1 accuracy BEFORE training: {acc_before:.0%}")
print(f"top-1 accuracy AFTER training:  {acc_after:.0%}")
print(f"improvement: +{(acc_after-acc_before)*100:.0f} percentage points")
```

🎉 **You understand how embedding models are trained!** From fixed hand-designed vectors to a
learned semantic space: contrastive loss, triplet loss, in-batch negatives (the efficiency trick
that makes large-scale training practical), the full training loop, retrieval-based evaluation,
and hard negative mining. This is exactly how Sentence-BERT, OpenAI's embedding models, and
every modern retrieval embedding is trained — just at massive scale with billions of real pairs.

---
### 📌 Concept Quick-Reference
**Goal:** learn text -> vector so similar meaning = close in space, different meaning = far apart
**Contrastive loss:** pull positive pairs together, push negative pairs apart past a margin
**Triplet loss:** relative constraint — negative must be further from anchor than positive, by a margin
**In-batch negatives:** free negatives from other pairs in the same batch (diagonal = correct matches)
**Training loop:** same 5-step rhythm as any PyTorch model, with a similarity-based loss
**Evaluation:** retrieval accuracy (top-1/top-k) — does the true match rank highest?
**Hard negative mining:** deliberately find confusingly-similar wrong matches to sharpen learning
**Real models:** Sentence-BERT, OpenAI embeddings — same ideas, billions of training pairs, huge batches
